In [ ]:
import folium
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
from skgstat import Variogram
import geopandas as gpd
from shapely.geometry import Point
import statsmodels.api as sm
import ast
from shapely.ops import unary_union
from pykrige.ok import OrdinaryKriging
from esda import G_Local
import matplotlib.pyplot as plt
from libpysal.weights import Kernel
import os
import sys
PROJECT_ROOT = os.path.abspath("..")  
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
sys.path.append(os.path.abspath("../models"))
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from esda.moran import Moran_Local
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from patsy import dmatrices
from pykrige.rk import RegressionKriging
from statsmodels.robust.scale import mad
from libpysal.weights import Kernel
from models.gwrmodel import GWRModel
from models.rfrkModel import RegressionKrigingModel
from models.sarModel import SpatialAutoregressiveModel
from models.modelEvaluator import ModelEvaluator
from models.spatialOutlierDetector import SpatialOutlierDetector
from models.strategies import *



In [ ]:
df = pd.read_csv("../storage/data/arg_venta_caba_processed.csv")

gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df.longitud, df.latitud),
    crs="EPSG:4326"
)
barrios = gpd.read_file("../barrios.geojson")


In [ ]:

STRATEGIES = {
    "negative": NegativeResidualsStrategy(),
    "ztest": ZTestStrategy(),
    "quantile": QuantileStrategy(),
    "lisa": LISAStrategy()
}
def detect_model_outliers(
    *,
    model,
    residuals,
    gdf,
    coords,
    output_dir,
    methods=None,
    k_neighbors=15,
    robust=False,
    lower_q=0.05,
    upper_q=0.95
):

    os.makedirs(output_dir, exist_ok=True)

    if methods is None:
        methods = ["negative", "ztest", "quantile", "lisa"]

    model_name = model.__class__.__name__
    res = residuals
    coords = model.coords if hasattr(model, "coords") else coords

    detector = SpatialOutlierDetector()

    results = {}

    w = None
    if any(m in methods for m in ["ztest", "lisa"]):

        w = Kernel(
            coords,
            k=k_neighbors,
            function="gaussian",
            fixed=False
        )

    for method in methods:

        strategy = STRATEGIES[method]

        result = strategy.run(
            res=res,
            gdf=gdf,
            coords=coords,
            detector=detector,
            w=w,
            robust=robust,
            lower_q=lower_q,
            upper_q=upper_q,
            output_dir=output_dir,
            model_name=model_name
        )

        results[method] = result

    return results

def run_outlier_analysis_for_models(
    *,
    models: list,
    coords,
    gdf,
    features,
    response,
    output_dir: str,
    methods=['lisa']
):
    """
    Corre detección de outliers para múltiples modelos.

    Parameters
    ----------
    models : list
        Lista de instancias de modelos ya entrenados.
    X, y, coords : datos
    gdf : GeoDataFrame original
    output_dir : carpeta base para guardar resultados
    """

    all_results = {}
    residuals_models = {}

    # -------------------------------------------------
    # 1️⃣ Calcular residuos para cada modelo
    # -------------------------------------------------
    for i in range(len(models)):
        model = models[i]
        f = features[i]
        
        model_name = model.__class__.__name__

        X = gdf[f]
        y = gdf[response].to_numpy().reshape(-1,1)

        print(f"Calculando residuos para {model_name}...")

        y_pred = model.in_sample_predictions()
        y = np.asarray(y).ravel()
        y_pred = np.asarray(y_pred).ravel()

        residuals = y - y_pred

        
        residuals = y - y_pred

        residuals_models[model_name] = residuals

    # -------------------------------------------------
    # 2️⃣ Detectar outliers para cada modelo
    # -------------------------------------------------
    for i in range(len(models)):

        model_name = models[i].__class__.__name__
        print(f"Detectando outliers para {model_name}...")

        results = detect_model_outliers(
            model=models[i],
            residuals=residuals_models[model_name],
            gdf=gdf,
            coords=coords,
            output_dir=f"{output_dir}/{model_name}",
            methods=methods
        )

        all_results[model_name] = results

    return all_results

In [ ]:

GWRmodel = GWRModel()
RFRKmodel = RegressionKrigingModel()
SARmodel = SpatialAutoregressiveModel()
features_gwr = [
    'area_m2_total',
    'area_m2_descubierta',
    'ambientes',
    'antiguedad',
    'expensas',
    'estado_num'
]

features_rfrk = [
    'area_m2_total',
    'area_m2_descubierta',
    'ambientes',
    'antiguedad',
    'expensas',
    'banos',
    'cocheras',
    'estado_num'
]

features_SAR = [
    'area_m2_total',
    'area_m2_descubierta',
    'ambientes',
    'antiguedad',
    'expensas',
    'banos',
    'cocheras',
    'estado_num'
]
features_for_models = [features_gwr, features_rfrk, features_SAR]

response = 'log_precio'

models = [GWRmodel, RFRKmodel, SARmodel]
metrics = {}
residuals_models = {}

gdf = gdf.to_crs(epsg=3857)

coords_tot   = np.column_stack([gdf.geometry.x,   gdf.geometry.y])


evaluator = ModelEvaluator()
for i in range(len(models)):
    model  = models[i]
    features = features_for_models[i]
    X_tot   = gdf[features]
    Y_tot   = gdf[response].to_numpy().reshape(-1,1)

    model.fit(X_tot, Y_tot, coords_tot)
    residuals_models[str(model.__class__.__name__)] = Y_tot - model.in_sample_predictions()
    


In [ ]:
gdf = gdf.reset_index(drop=True)
coords = np.array([(geom.x, geom.y) for geom in gdf.geometry])


response = 'log_precio'

models = [GWRmodel, RFRKmodel, SARmodel]
metrics = {}
residuals_models = {}


results = run_outlier_analysis_for_models(
    models=models,
    coords=coords,
    gdf=gdf,
    features=features_for_models,
    response = response,
    output_dir="output/venta"
)